In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# Giải nén tập dữ liệu 
%cd /content
!unzip -q /content/drive/MyDrive/EdgeCard_System/dataset_det.zip

In [ ]:
# Python mặc định Colab
%cd /content

!python --version

# Paddle GPU - cùng môi trường với PP-OCRv6 Recognition
!python -m pip install -q paddlepaddle-gpu==3.2.1 \
    -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# PaddleOCR
!rm -rf /content/PaddleOCR
!git clone -q --depth 1 --branch v3.7.0 https://github.com/PaddlePaddle/PaddleOCR.git

%cd /content/PaddleOCR
!python -m pip install -q -r requirements.txt


In [ ]:
# Kiểm tra môi trường
import sys, paddle
print("Python :", sys.version.split()[0])
print("Paddle :", paddle.__version__)
print("CUDA   :", paddle.version.cuda())
print("GPU    :", paddle.device.is_compiled_with_cuda())
paddle.utils.run_check()


In [ ]:
%cd /content/PaddleOCR

# ============================================================
# 1. Chuẩn bị pretrained PP-OCRv6 Small Det
# ============================================================

!mkdir -p pretrain_models

!wget -nc -P pretrain_models/ \
https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_small_det_pretrained.pdparams


# ============================================================
# 2. Tạo config PP-OCRv6 Small Det
# ============================================================

import os

os.makedirs("configs/det", exist_ok=True)

yaml_content = """Global:
  model_name: PP-OCRv6_small_det
  debug: false

  use_gpu: true
  use_xpu: false
  use_mlu: false

  use_ema: true
  ema_decay: 0.9997
  ema_decay_type: threshold

  # Tăng từ 50 -> 100 để PP-OCRv6 có đủ thời gian fine-tune
  epoch_num: &epoch_num 100

  log_smooth_window: 20
  print_batch_step: 10

  # Lưu folder mới để không ghi đè model cũ
  save_model_dir: /content/drive/MyDrive/EdgeCard_System/stage_2/pp-ocrv6_v2

  save_epoch_step: 10

  # Khoảng 1 lần eval / epoch với batch hiện tại
  eval_batch_step: [0, 122]

  cal_metric_during_train: false

  pretrained_model: ./pretrain_models/PP-OCRv6_small_det_pretrained.pdparams

  checkpoints:
  save_inference_dir:

  use_visualdl: false

  infer_img:

  save_res_path: /content/drive/MyDrive/EdgeCard_System/stage_2/pp-ocrv6_v2/predicts_db.txt

  d2s_train_image_shape: [3, 640, 640]

  distributed: false


Architecture:
  model_type: det
  algorithm: DB

  Transform:

  Backbone:
    name: PPLCNetV4
    det: true
    model_size: small

  Neck:
    name: RepLKFPN
    out_channels: 96
    dilated_kernel_size: 7
    shortcut: true

  Head:
    name: DBHead
    k: 50
    fix_nan: true
    aux_in_channels: 96


Loss:
  name: DBLoss

  main_loss_type: DiceFocalLoss

  alpha: 5
  beta: 10

  focal_alpha: 0.25
  focal_gamma: 2.5

  aux_weight_p4: 0.2
  aux_weight_p3: 0.3
  aux_weight_p2: 0.4


Optimizer:
  name: Adam

  beta1: 0.9
  beta2: 0.999

  lr:
    name: Cosine
    learning_rate: 0.001
    warmup_epoch: 2

  regularizer:
    name: L2
    factor: 1.0e-05


# ============================================================
# PostProcess
# Giữ nguyên cấu hình chuẩn lúc train.
# Sau khi có best_accuracy mới tune trên VALIDATION SET.
# ============================================================

PostProcess:
  name: DBPostProcess

  thresh: 0.2
  box_thresh: 0.45

  max_candidates: 3000

  unclip_ratio: 1.4


Metric:
  name: DetMetric
  main_indicator: hmean


# ============================================================
# TRAIN
# ============================================================

Train:
  dataset:
    name: SimpleDataSet

    data_dir: /content/dataset_det/

    label_file_list:
      - /content/dataset_det/train_label.txt

    ratio_list: [1.0]

    transforms:

      # ------------------------------------------------------
      # Decode
      # ------------------------------------------------------
      - DecodeImage:
          img_mode: BGR
          channel_first: false

      - DetLabelEncode:


      # ------------------------------------------------------
      # Data augmentation
      #
      # Không dùng Fliplr vì text trên thẻ không xuất hiện
      # dưới dạng mirror trong thực tế.
      #
      # Không xoay +/-45 độ như ICDAR vì ảnh thẻ đã được
      # Stage 1 perspective alignment.
      # ------------------------------------------------------

      - IaaAugment:
          augmenter_args:

            # Xoay nhẹ phù hợp ảnh sau khi rectification
            - type: Affine
              args:
                p: 0.5
                rotate: [-10, 10]
                fit_output: true

            # Thêm scale augmentation nhẹ
            - type: Resize
              args:
                size: [0.8, 1.2]


      # ------------------------------------------------------
      # Dùng RandomCrop theo recipe PP-OCRv6
      # ------------------------------------------------------

      - RandomCrop:
          size: [640, 640]
          max_tries: 50
          keep_ratio: true


      # ------------------------------------------------------
      # DB maps
      # ------------------------------------------------------

      - MakeBorderMap:
          shrink_ratio: 0.4

          thresh_min: 0.3
          thresh_max: 0.7

          total_epoch: *epoch_num


      - MakeShrinkMap:
          shrink_ratio: 0.4

          min_text_size: 8

          total_epoch: *epoch_num


      # ------------------------------------------------------
      # Normalize
      # ------------------------------------------------------

      - NormalizeImage:
          scale: 1./255.

          mean:
            - 0.485
            - 0.456
            - 0.406

          std:
            - 0.229
            - 0.224
            - 0.225

          order: hwc


      - ToCHWImage:


      - KeepKeys:
          keep_keys:
            - image
            - threshold_map
            - threshold_mask
            - shrink_map
            - shrink_mask


  loader:

    shuffle: true

    drop_last: false

    # Giữ batch=8 vì PP-OCRv6 nặng hơn v4/DBNet
    batch_size_per_card: 8

    num_workers: 2

    use_shared_memory: false


# ============================================================
# VALIDATION
# ============================================================

Eval:
  dataset:

    name: SimpleDataSet

    data_dir: /content/dataset_det/

    label_file_list:
      - /content/dataset_det/valid_label.txt

    transforms:

      - DecodeImage:
          img_mode: BGR
          channel_first: false

      - DetLabelEncode:


      # ------------------------------------------------------
      # Giữ 736x1280 giống PP-OCRv4 và DBNet
      # để benchmark công bằng
      # ------------------------------------------------------

      - DetResizeForTest:
          image_shape: [736, 1280]


      - NormalizeImage:
          scale: 1./255.

          mean:
            - 0.485
            - 0.456
            - 0.406

          std:
            - 0.229
            - 0.224
            - 0.225

          order: hwc


      - ToCHWImage:


      - KeepKeys:
          keep_keys:
            - image
            - shape
            - polys
            - ignore_tags


  loader:

    shuffle: false

    drop_last: false

    batch_size_per_card: 1

    num_workers: 2

    use_shared_memory: false


profiler_options: null
"""


# ============================================================
# 3. Ghi config
# ============================================================

CONFIG_PATH = "configs/det/custom_ppocrv6_v2.yml"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("=" * 70)
print("Đã tạo config PP-OCRv6 Small Det mới")
print("Config:", CONFIG_PATH)
print("=" * 70)


# ============================================================
# 4. Kiểm tra nhanh config
# ============================================================

import yaml

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("Model        :", cfg["Global"]["model_name"])
print("Epoch        :", cfg["Global"]["epoch_num"])
print("Batch        :", cfg["Train"]["loader"]["batch_size_per_card"])
print("Train shape  :", cfg["Global"]["d2s_train_image_shape"])
print(
    "Eval shape   :",
    cfg["Eval"]["dataset"]["transforms"][2]["DetResizeForTest"]["image_shape"]
)
print("Learning rate:", cfg["Optimizer"]["lr"]["learning_rate"])
print("box_thresh   :", cfg["PostProcess"]["box_thresh"])
print("unclip_ratio :", cfg["PostProcess"]["unclip_ratio"])

print("\nConfig OK!")

In [ ]:
%cd /content/PaddleOCR

!python tools/train.py -c configs/det/custom_ppocrv6_v2.yml

In [ ]:
%cd /content/PaddleOCR

# Đánh giá PP-OCRv6 Small Det trên test set
!python tools/eval.py \
    -c configs/det/custom_ppocrv6_v2.yml \
    -o Global.checkpoints=/content/drive/MyDrive/EdgeCard_System/stage_2/pp-ocrv6_v2/best_accuracy \
       Eval.dataset.label_file_list=["/content/dataset_det/test_label.txt"]
